In [1]:
# Install pygeohash silently

import warnings
import random
import lightgbm as lgb
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.metrics import r2_score
from pathlib import Path


warnings.filterwarnings('ignore')

# --- 1. CONFIGURATION ---
TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'
SUBMISSION_PATH = Path('Submission files/submission_codex_scaled.csv')

# UPGRADE 1: Expanded Lags for a "24-Hour Window" instead of a single point
LAG_FEATURES = [1, 2, 3, 4, 8, 12, 24, 95, 96, 97]
ROLLING_WINDOWS = [4, 8, 12, 24]
DYNAMIC_FEATURE_COLUMNS = [
    *[f'demand_lag_{lag}' for lag in LAG_FEATURES],
    *[f'demand_roll_mean_{window}' for window in ROLLING_WINDOWS],
    *[f'demand_roll_std_{window}' for window in ROLLING_WINDOWS],
]

# --- 2. DATA PREPARATION (Pure Original Logic) ---
def parse_time(series):
    parts = series.astype(str).str.split(':', n=1, expand=True)
    return parts[0].astype(int) * 60 + parts[1].astype(int)

def decode_geo(val):
    if pd.isna(val): return np.nan, np.nan
    try:
        dec = pgh.decode(str(val))
        return float(dec[0]), float(dec[1])
    except: return np.nan, np.nan

def add_geo_coords(df):
    geos = df['geohash'].astype('string').dropna().unique()
    lookup = {v: decode_geo(v) for v in geos}
    coords = df['geohash'].astype('string').map(lookup)
    df['latitude'] = [p[0] if isinstance(p, tuple) else np.nan for p in coords]
    df['longitude'] = [p[1] if isinstance(p, tuple) else np.nan for p in coords]
    return df

def build_features(df):
    grp = df.groupby('geohash', sort=False)
    for lag in LAG_FEATURES:
        df[f'demand_lag_{lag}'] = grp['demand'].shift(lag)
    lag_grp = grp['demand'].shift(1).groupby(df['geohash'], sort=False)
    for w in ROLLING_WINDOWS:
        df[f'demand_roll_mean_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).mean())
        df[f'demand_roll_std_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).std())
    df[DYNAMIC_FEATURE_COLUMNS] = df[DYNAMIC_FEATURE_COLUMNS].fillna(df.groupby('geohash')['demand'].transform('mean'))
    return df

def get_dyn_feats(hist):
    feats = {}
    for lag in LAG_FEATURES:
        feats[f'demand_lag_{lag}'] = hist[-lag] if len(hist) >= lag else np.nan
    for w in ROLLING_WINDOWS:
        vals = hist[-w:]
        if vals:
            feats[f'demand_roll_mean_{w}'] = float(np.mean(vals))
            feats[f'demand_roll_std_{w}'] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        else:
            feats[f'demand_roll_mean_{w}'] = np.nan
            feats[f'demand_roll_std_{w}'] = np.nan
    return feats

# --- 3. RECURSIVE INFERENCE (No Log/Exp, Pure Raw Demand) ---
def recursive_predict(model, hist_df, tgt_df, feat_cols, cat_cols, cat_levels):
    if tgt_df.empty: return pd.Series(dtype=float)
    stat_cols = [c for c in feat_cols if c not in DYNAMIC_FEATURE_COLUMNS]
    preds, p_idx = [], []
    
    hist_map = {str(g): grp.sort_values(['day', 'time_mins'])['demand'].astype(float).tolist()
                for g, grp in hist_df.groupby('geohash', sort=False)}

    for g_val, g_grp in tgt_df.sort_values(['geohash', 'day', 'time_mins']).groupby('geohash', sort=False):
        g_key = str(g_val)
        h_vals = hist_map.get(g_key, [])
        
        for row in g_grp.itertuples(index=False):
            f_row = {c: getattr(row, c) for c in stat_cols}
            f_row.update(get_dyn_feats(h_vals))
            f_df = pd.DataFrame([f_row], columns=feat_cols)
            for c in cat_cols: f_df[c] = pd.Categorical(f_df[c], categories=cat_levels[c])
            
            p = float(np.asarray(model.predict(f_df))[0])
            p = max(0.0, p)
            
            preds.append(p)
            p_idx.append(int(getattr(row, 'Index')))
            h_vals.append(p)

    return pd.Series(preds, index=p_idx)

def align_predictions(pred_series, index_list):
    mapping = {int(k): float(v) for k, v in pred_series.to_dict().items()}
    preds = np.array([mapping.get(idx, np.nan) for idx in index_list], dtype=float)
    if np.isnan(preds).any():
        preds = np.where(np.isnan(preds), 0.0, preds)
    return preds

# --- 4. EXECUTION ---
print("1. Loading and Structuring Data...")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_raw['is_test'] = False
test_raw['is_test'] = True
test_raw['demand'] = np.nan

df = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)
df['time_mins'] = parse_time(df['timestamp']).astype(int)
df['time_sin'] = np.sin(2 * np.pi * df['time_mins'] / 1440.0)
df['time_cos'] = np.cos(2 * np.pi * df['time_mins'] / 1440.0)
df = add_geo_coords(df)
df = df.sort_values(['geohash', 'day', 'time_mins']).reset_index(drop=True)

df['Temperature'] = df.groupby('geohash')['Temperature'].transform(lambda s: s.ffill().bfill())
df['Weather'] = df.groupby('geohash')['Weather'].transform(lambda s: s.ffill().bfill())
df['RoadType'] = df['RoadType'].fillna('Unknown')
df = build_features(df)

cat_cols = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_cols: df[c] = df[c].astype('category')
cat_levels = {c: df[c].cat.categories for c in cat_cols}

train_df = df[~df['is_test']].copy()
test_df = df[df['is_test']].copy()

f_cols = ['geohash', 'day', 'time_mins', 'time_sin', 'time_cos', 'latitude', 'longitude', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'] + DYNAMIC_FEATURE_COLUMNS

train_mask = (train_df['day'] == 48)
valid_mask = (train_df['day'] == 49) & (train_df['time_mins'] <= 120)

X_train = train_df.loc[train_mask, f_cols].copy()
y_train = train_df.loc[train_mask, 'demand'].astype(float)
X_valid = train_df.loc[valid_mask, f_cols].copy()
y_valid = train_df.loc[valid_mask, 'demand'].astype(float)

# --- 5. MASSIVE RANDOM SEARCH ---
print("\n2. Executing Massive Random Search (40 Trials)...")
N_TRIALS = 40
TOP_K = 10 # UPGRADE 2: Ensembling 10 models for maximum stability

def sample_params(rng):
    return {
        'learning_rate': rng.choice([0.01, 0.02, 0.03, 0.05, 0.07]),
        'num_leaves': rng.choice([63, 127, 255]),
        'min_child_samples': rng.choice([5, 10, 20, 30, 50]),
        'subsample': rng.choice([0.7, 0.8, 0.9, 1.0]),
        'colsample_bytree': rng.choice([0.7, 0.8, 0.9, 1.0]),
        'reg_alpha': rng.choice([0.01, 0.1, 0.5, 1.0]),
        'reg_lambda': rng.choice([0.01, 0.1, 0.5, 1.0]),
    }

rng = random.Random(999)
trials = []

for i in range(N_TRIALS):
    params = sample_params(rng)
    
    # UPGRADE 3: Native GPU Execution with max_bin fix
    try:
        model = lgb.LGBMRegressor(
            objective='regression', metric='rmse', verbosity=-1,
            device_type='gpu', max_bin=255, n_estimators=1500, random_state=i, **params
        )
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(50, verbose=False)])
    except lgb.basic.LightGBMError:
        model = lgb.LGBMRegressor(
            objective='regression', metric='rmse', verbosity=-1,
            device_type='cpu', n_jobs=-1, n_estimators=1500, random_state=i, **params
        )
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(50, verbose=False)])

    val_hist = train_df.loc[train_df['day'] == 48].copy()
    val_target = train_df.loc[valid_mask].copy()
    
    val_series = recursive_predict(model, val_hist, val_target, f_cols, cat_cols, cat_levels)
    val_preds = align_predictions(val_series, val_target['Index'].astype(int).tolist())
    val_r2 = r2_score(y_valid, val_preds)

    trials.append({'model': model, 'r2': val_r2})
    print(f"   -> Trial {i+1}/{N_TRIALS}: Validation R2 = {val_r2:.6f}")

# --- 6. SUPER-ENSEMBLE GENERATION ---
print("\n3. Generating Final 10-Model Super-Ensemble on Test Set...")
trials_sorted = sorted(trials, key=lambda x: x['r2'], reverse=True)
top_trials = trials_sorted[:TOP_K]

idx_list = test_df['Index'].astype(int).tolist()
ensemble_test = np.zeros(len(idx_list), dtype=float)

for idx, t in enumerate(top_trials):
    print(f"   -> Ensembling Model {idx+1}...")
    s = recursive_predict(t['model'], train_df.copy(), test_df.copy(), f_cols, cat_cols, cat_levels)
    ensemble_test += align_predictions(s, idx_list)

ensemble_test /= len(top_trials)
submission_ensemble = pd.DataFrame({'Index': idx_list, 'demand': ensemble_test})
submission_ensemble.to_csv(SUBMISSION_PATH, index=False)

print(f'\n--- 🏆 SCALED CODEX SUPER-ENSEMBLE READY 🏆 ---')
print(f'Wrote {SUBMISSION_PATH}')

1. Loading and Structuring Data...

2. Executing Massive Random Search (40 Trials)...
   -> Trial 1/40: Validation R2 = 0.909940
   -> Trial 2/40: Validation R2 = 0.918668
   -> Trial 3/40: Validation R2 = 0.917372
   -> Trial 4/40: Validation R2 = 0.920195
   -> Trial 5/40: Validation R2 = 0.919518
   -> Trial 6/40: Validation R2 = 0.910216
   -> Trial 7/40: Validation R2 = 0.923426
   -> Trial 8/40: Validation R2 = 0.920859
   -> Trial 9/40: Validation R2 = 0.913182
   -> Trial 10/40: Validation R2 = 0.916773
   -> Trial 11/40: Validation R2 = 0.913916
   -> Trial 12/40: Validation R2 = 0.918568
   -> Trial 13/40: Validation R2 = 0.914487
   -> Trial 14/40: Validation R2 = 0.909668
   -> Trial 15/40: Validation R2 = 0.915157
   -> Trial 16/40: Validation R2 = 0.904843
   -> Trial 17/40: Validation R2 = 0.914082
   -> Trial 18/40: Validation R2 = 0.916159
   -> Trial 19/40: Validation R2 = 0.913318
   -> Trial 20/40: Validation R2 = 0.907992
   -> Trial 21/40: Validation R2 = 0.920697

In [3]:
import pandas as pd
from sklearn.metrics import r2_score

actual_df = pd.read_csv('submission.csv').sort_values('Index').reset_index(drop=True)
pred_df = pd.read_csv(SUBMISSION_PATH).sort_values('Index').reset_index(drop=True)

r2 = r2_score(actual_df['demand'], pred_df['demand'])
print(f"🚀 EVALUATION 🚀")
print(f"True Leaderboard Score: {max(0, 100 * r2):.4f} / 100")

🚀 EVALUATION 🚀
True Leaderboard Score: 90.6350 / 100
